# Lecture 31 - Reproducible Workflows and Best Practices

## Learning Objectives

- Structure data science projects with a standard directory layout
- Use Git for version control in data science projects
- Manage dependencies with virtual environments and requirements files
- Write unit tests for data pipelines with pytest
- Implement logging for data processing steps
- Apply notebook best practices for reproducible research

## Key Topics

- Project directory structure (Cookiecutter Data Science)
- Git basics: init, add, commit, push, branching
- Virtual environments: conda env and venv
- requirements.txt and environment.yml
- Unit testing with pytest for data pipelines
- Logging with the logging module
- Notebook best practices

## Project Directory Structure

A consistent project structure makes your work understandable to collaborators (and your future self). The **Cookiecutter Data Science** template is the de-facto standard:

```
project/
├── data/
│   ├── raw/          # immutable original data
│   ├── processed/    # cleaned, transformed data
│   └── interim/      # intermediate results
├── notebooks/        # exploratory notebooks
├── src/              # reusable Python modules
├── tests/            # unit tests
├── models/           # serialised model files
├── reports/          # generated reports and figures
├── requirements.txt  # project dependencies
├── environment.yml   # conda environment file
├── README.md         # project overview
└── .gitignore        # files git should ignore
```

This separation of raw, processed, and interim data ensures you always know which data is original and which is derived. Notebooks go in their own folder, reusable code goes in `src/`, and models are stored separately so they can be loaded for inference.

In [ ]:
# Example: programmatically create the directory structure
import os
from pathlib import Path

base = Path("/tmp/my_project")
dirs = [
    "data/raw", "data/processed", "data/interim",
    "notebooks", "src", "tests", "models", "reports/figures"
]
for d in dirs:
    (base / d).mkdir(parents=True, exist_ok=True)

print("Project structure created at", base)
for p in sorted(base.rglob("*")):
    if p.is_dir():
        print(f"  {p.relative_to(base)}/")

## Git Basics for Data Scientists

Version control is not just for software engineers. As a data scientist, you need Git to:

- Track changes to code, notebooks, and configuration files
- Experiment with different approaches on branches
- Collaborate with teammates without overwriting work
- Revert to a previous version when something breaks

Essential commands:
- `git init` — initialise a repository
- `git add <file>` — stage changes
- `git commit -m "message"` — commit staged changes
- `git push` — upload to remote (GitHub/GitLab)
- `git branch <name>` — create a branch for experimentation
- `git checkout <branch>` — switch branches

**Never** commit large data files or credentials. Use `.gitignore` to exclude data files and `.env` files from version control.

In [ ]:
# Git commands shown as strings (run in terminal, not here)
commands = 
print("Git workflow:")
for line in commands.strip().split("\n"):
    if line.strip():
        print(f"  $ {line.strip()}")

In [ ]:
# Example .gitignore
ignore_rules = [
    "# Data files",
    "*.csv", "*.xlsx", "*.parquet",
    "data/raw/*", "data/processed/*",
    "",
    "# Environment and secrets",
    ".env", "*.env.local",
    "",
    "# Notebook outputs",
    ".ipynb_checkpoints/", "*.nbconvert.*",
    "",
    "# Python cache",
    "__pycache__/", "*.pyc", "*.eggs",
    "",
    "# Models",
    "*.pkl", "*.h5", "models/",
]
print("Recommended .gitignore:")
print("\n".join(ignore_rules))

## Virtual Environments and Dependency Management

A **virtual environment** isolates your project's dependencies so different projects can use different library versions without conflicts.

```bash
# Using venv (built-in)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

# Using conda
conda env create -f environment.yml
conda activate my_project
```

Two files capture dependencies:
- **requirements.txt**: flat list of `pip` packages pinning exact versions
- **environment.yml**: conda's richer format supporting channels and conda packages

Always pin exact versions (e.g., `pandas==2.0.3`) rather than loose versions (`pandas>=2.0`) so that anyone can recreate your exact environment.

In [ ]:
# Example requirements.txt as a list
req_lines = [
    "pandas==2.0.3", "numpy==1.25.2", "scikit-learn==1.3.0",
    "matplotlib==3.7.2", "jupyter==1.0.0", "pytest==7.4.0",
    "black==23.7.0", "flake8==6.1.0",
]
print("Example requirements.txt:")
print("\n".join(req_lines))

In [ ]:
# Example environment.yml as a multi-line string
env_lines = [
    "name: ds_project",
    "channels:",
    "  - conda-forge",
    "  - defaults",
    "dependencies:",
    "  - python=3.10",
    "  - pandas=2.0.3",
    "  - numpy=1.25.2",
    "  - scikit-learn=1.3.0",
    "  - matplotlib=3.7.2",
    "  - jupyter=1.0.0",
    "  - pip",
    "  - pip:",
    "    - pytest==7.4.0",
    "    - black==23.7.0",
    "    - flake8==6.1.0",
]
print("Example environment.yml:")
print("\n".join(env_lines))

## Writing Unit Tests with pytest

Data pipelines are software, and software should be tested. **Unit tests** verify that individual components (functions, transformations) produce correct outputs.

pytest is the most popular testing framework for Python. Test files go in a `tests/` directory, and test functions start with `test_`. A simple assertion is all you need:

```python
def test_clean_column_names():
    df = pd.DataFrame({"My Column": [1, 2]})
    result = clean_column_names(df)
    assert "my_column" in result.columns
```

Testing data pipelines catches subtle bugs: off-by-one errors, wrong column types, incorrect filtering logic. It may feel slow at first, but it saves hours of debugging later.

In [ ]:
# Define a simple data processing function
def clean_column_names(df):
    import pandas as pd
    df_clean = df.copy()
    df_clean.columns = [
        col.strip().lower().replace(" ", "_") for col in df_clean.columns
    ]
    return df_clean

def clip_outliers(series, lower=0.01, upper=0.99):
    q_low = series.quantile(lower)
    q_high = series.quantile(upper)
    return series.clip(q_low, q_high)

# Test the functions
import pandas as pd
import numpy as np

test_df = pd.DataFrame({"A Col": [10, 20], "B Col": [30, 40]})
cleaned = clean_column_names(test_df)
print("Cleaned columns:", list(cleaned.columns))
assert list(cleaned.columns) == ["a_col", "b_col"]

series = pd.Series([1, 2, 3, 100, 200, 5])
clipped = clip_outliers(series, 0.1, 0.9)
print(f"Clipped series: min={clipped.min()}, max={clipped.max()}")
assert clipped.max() <= series.quantile(0.9)
assert clipped.min() >= series.quantile(0.1)
print("All tests passed!")

In [ ]:
# Show how tests would be structured in a file
test_script = '''
# tests/test_data_utils.py
import pandas as pd
import numpy as np
from src.data_utils import clean_column_names, clip_outliers

def test_clean_column_names():
    df = pd.DataFrame({"A Col": [1, 2], "B Col": [3, 4]})
    result = clean_column_names(df)
    assert list(result.columns) == ["a_col", "b_col"]

def test_clip_outliers_bounds():
    s = pd.Series([1, 2, 3, 100, 200])
    result = clip_outliers(s, 0.1, 0.9)
    assert result.min() >= s.quantile(0.1)
    assert result.max() <= s.quantile(0.9)

def test_clip_outliers_preserves_middle():
    s = pd.Series([1, 2, 3, 4, 5, 100])
    result = clip_outliers(s, 0.1, 0.9)
    assert result.iloc[2] == 3
'''
print("Example test file (tests/test_data_utils.py):")
print(test_script)

## Logging with the logging Module

Print statements work for quick debugging, but they are not suitable for production pipelines. The **logging** module provides:

- **Log levels**: DEBUG, INFO, WARNING, ERROR, CRITICAL
- **Output control**: route logs to console, files, or both
- **Timestamps**: every log entry gets a timestamp automatically
- **Granularity**: turn debug logs on/off without editing code

A typical data pipeline logs each step: "Loading data...", "Cleaning data...found 45 missing values", "Feature engineering...created 12 features", "Training model...CV score=0.87".

This creates an audit trail that makes debugging and monitoring much easier.

In [ ]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("/tmp/pipeline.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Simulate a data pipeline with logging
logger.info("Starting data pipeline")

logger.info("Loading data from CSV...")
df_sim = pd.DataFrame(np.random.rand(100, 3), columns=list("abc"))

logger.info(f"Loaded {len(df_sim)} rows with {len(df_sim.columns)} columns")

missing_count = df_sim.isnull().sum().sum()
if missing_count > 0:
    logger.warning(f"Found {missing_count} missing values")
else:
    logger.info("No missing values found")

logger.info("Feature engineering step complete")
logger.info("Pipeline finished successfully")

# Show log file content
with open("/tmp/pipeline.log") as f:
    print("\nLog file contents:")
    print(f.read())

## Notebook Best Practices

Jupyter notebooks are the most popular tool for exploratory data analysis, but they can easily become messy. Follow these best practices:

1. **Run top-to-bottom**: always execute cells in order before sharing. Restart the kernel and "Run All" to verify.
2. **Keep cells small**: each cell should do one thing — load data, clean data, visualise, etc.
3. **Use functions**: encapsulate reusable logic in functions, defined at the top of the notebook or in a separate `.py` module.
4. **Clear outputs**: strip cell outputs before committing to Git (use `jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace notebook.ipynb`).
5. **Use markdown cells**: document your thinking, explain your decisions, and present results in markdown, not code comments.
6. **Version control notebooks**: Jupyter diffs are messy but tools like `nbdime` and ReviewNB make code review possible.
7. **Avoid state-dependence**: don't rely on executing cells in a non-linear order. Use `%load_ext autoreload` to pick up changes from `.py` modules.

In [ ]:
# Demonstration of notebook best practices
# 1. Imports at the top
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 2. Reusable functions defined early
def compute_summary_stats(df):
    return pd.DataFrame({
        "mean": df.mean(),
        "std": df.std(),
        "min": df.min(),
        "max": df.max()
    })

# 3. Then data loading, cleaning, analysis
np.random.seed(42)
df_demo = pd.DataFrame(
    np.random.randn(100, 3),
    columns=["feature_a", "feature_b", "feature_c"]
)

# 4. Analysis and visualisation
summary = compute_summary_stats(df_demo)
print("Summary statistics:")
print(summary)

df_demo.hist(bins=20, figsize=(10, 4))
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()

## Data Science Connection

Reproducibility is the cornerstone of scientific computing. A project that cannot be reproduced — by you, six months from now, on a different machine — is not trustworthy. The practices in this lecture — project structure, version control, dependency management, testing, logging, and notebook discipline — transform ad-hoc analyses into professional, auditable data science. Every top data science team follows these practices, and adopting them early will save you countless hours of frustration.